# Density estimation benchmark

Run `python -m data.preprocess.generate` before this notebook. It evaluates GEB, PT, and variational DPMM on all six datasets and skips completed method/replication results.

Run HMPT separately using `01_density_estimation_hmpt.Rmd`, which uses the R package `SMCMP` and updates the same result CSVs. Run the Python and R notebooks sequentially.

Each result CSV includes means and standard errors. GEB timing summaries omit replication 1 to reduce the effect of initial JAX compilation; scores and other timings use all 20 replications.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'experiment_utils').exists():
    raise FileNotFoundError('Open this notebook from the project root or notebooks folder.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from IPython.display import display
from experiment_utils.config import DATASETS, TREE_TRANSFORMS
from experiment_utils.run_benchmark import run_dataset

# m is the candidate-grid size (m - 1 interior cuts per coordinate).
GEB_PARAMETERS = dict(m=32, alpha=1.0, grid=True)
PT_PARAMETERS = dict(max_depth=15, alpha_scale=0.1)
DPMM_PARAMETERS = dict(
    max_components=50, weight_concentration=1.0, max_iter=1_000,
    n_init=5, tol=1e-3, reg_covar=1e-6,
)
MODEL_PARAMETERS = {'GEB': GEB_PARAMETERS, 'PT': PT_PARAMETERS, 'DPMM': DPMM_PARAMETERS}
print('GEB:', GEB_PARAMETERS)
print('PT:', PT_PARAMETERS)
print('VB-DPMM (diagonal covariance, k-means initialization):', DPMM_PARAMETERS)
print('HMPT: max resolution=15, eta=0.1, grid=32, particles=1000; run the companion Rmd.')
DATASET_TRANSFORMS = pd.DataFrame([
    {'dataset': dataset, 'tree_transforms': ', '.join(transforms),
     'DPMM_scale': 'saved analysis coordinates'}
    for dataset, transforms in TREE_TRANSFORMS.items()
])
display(DATASET_TRANSFORMS)


## Run GEB, PT and VB-DPMM

In [ ]:
PYTHON_METHODS = ('GEB', 'PT', 'DPMM')

for dataset in DATASETS:
    print(f'\n===== {dataset} =====', flush=True)
    run_dataset(dataset, models=PYTHON_METHODS, parameters=MODEL_PARAMETERS)

print('\nPython density-estimation methods are complete.')
print('Now run 01_density_estimation_hmpt.Rmd for HMPT.')


## Result location

In [ ]:
RESULT_DIR = PROJECT_ROOT / 'results' / 'density_estimation'
print(RESULT_DIR)
pd.DataFrame({'file': sorted(path.name for path in RESULT_DIR.iterdir() if path.suffix in {'.csv', '.tex'})})
